# 03.3 Attention Intro / Attention 入门

`Attention` 是 Transformer 时代的核心概念之一。  
`Attention` is one of the core concepts of the Transformer era.

如果用一句话概括它的直觉：  
If we summarize its intuition in one sentence:

- 模型在处理一个位置时，会动态决定要“关注”序列里的哪些位置 / when processing one position, the model dynamically decides which other positions in the sequence to attend to

本节先不追求完整 Transformer，而是把最小 attention 机制看懂。  
This notebook does not aim to cover the full Transformer yet; it aims to make the minimal attention mechanism understandable.

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 理解 `query / key / value` 的基本角色 / Understand the basic roles of `query / key / value`.
2. 手动计算一个小型 attention 例子 / Manually compute a small attention example.
3. 理解 attention score、softmax 权重、context vector / Understand attention scores, softmax weights, and context vectors.
4. 理解为什么要除以 `sqrt(d_k)` / Understand why we divide by `sqrt(d_k)`.
5. 理解 mask 在 attention 中的作用 / Understand the role of masking in attention.
6. 用 `PyTorch` 写一个最小 self-attention 模块 / Write a minimal self-attention module in `PyTorch`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. 一个最小 attention 例子 / A Minimal Attention Example

注意力 / attention 的核心计算可以简写成：  
The core attention computation can be summarized as:

1. `scores = QK^T`
2. `weights = softmax(scores)`
3. `context = weights * V`

这里先从非常小的矩阵开始。  
We start with very small matrices here.

In [ ]:
Q = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
K = torch.tensor([[1.0, 0.0], [0.5, 1.0]])
V = torch.tensor([[10.0, 0.0], [0.0, 20.0]])

scores = Q @ K.T
weights = F.softmax(scores, dim=-1)
context = weights @ V

print("Q =\n", Q)
print("K =\n", K)
print("V =\n", V)
print("scores =\n", scores)
print("weights =\n", weights)
print("context =\n", context)

直觉理解 / Intuition:

- `Q`：当前“我要找什么” / what I am looking for right now
- `K`：每个位置“我有什么特征” / what features each position has
- `V`：每个位置真正要被加权汇总的内容 / the content that will actually be mixed together

注意：这只是最小直觉版本，不是唯一解释方式。  
Note: this is only the minimal intuition version, not the only valid interpretation.

## 2. 为什么要除以 `sqrt(d_k)`
## Why Divide by `sqrt(d_k)`?

当维度 `d_k` 增大时，点积的值往往也会变大。  
When the dimension `d_k` grows, dot-product values also tend to grow.

如果分数过大，`softmax` 会变得过于尖锐 / too sharp。  
If the scores become too large, the `softmax` can become overly sharp.

所以常见写法是：  
So the common formula is:

- `scores = QK^T / sqrt(d_k)`

In [ ]:
Q = torch.randn(2, 8)
K = torch.randn(2, 8)

raw_scores = Q @ K.T
scaled_scores = raw_scores / math.sqrt(Q.size(-1))

print("raw_scores =\n", raw_scores)
print("scaled_scores =\n", scaled_scores)
print("softmax(raw_scores) =\n", F.softmax(raw_scores, dim=-1))
print("softmax(scaled_scores) =\n", F.softmax(scaled_scores, dim=-1))

## 3. Self-Attention / Self-Attention

`Self-Attention` 的意思是：  
`Self-Attention` means:

- `Q`、`K`、`V` 都来自同一个输入序列 / `Q`, `K`, and `V` all come from the same input sequence

这是 Transformer 中最核心的情况。  
This is the core case used inside Transformers.

In [ ]:
x = torch.tensor(
    [
        [1.0, 0.0, 1.0],
        [0.0, 1.0, 1.0],
        [1.0, 1.0, 0.0],
    ]
)

Q = x
K = x
V = x

scores = (Q @ K.T) / math.sqrt(x.size(-1))
weights = F.softmax(scores, dim=-1)
context = weights @ V

print("x =\n", x)
print("scores =\n", scores)
print("weights =\n", weights)
print("context =\n", context)

这里每个位置都能“看”到序列中的其他位置。  
Here every position can "look at" other positions in the sequence.

这和 `LSTM` 的逐步传播方式很不一样。  
This is very different from the step-by-step propagation style of an `LSTM`.

## 4. Padding Mask / Padding Mask

如果某些位置只是 padding，就不应该让模型关注它们。  
If some positions are only padding, the model should not attend to them.

常见做法是把这些位置的 score 设成很小的负数。  
A common approach is to set the scores at those positions to a very large negative value.

In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.5]])
padding_mask = torch.tensor([[False, False, True]])

masked_scores = scores.masked_fill(padding_mask, float("-inf"))
weights = F.softmax(masked_scores, dim=-1)

print("scores =", scores)
print("masked_scores =", masked_scores)
print("weights =", weights)

这样最后一个 padding 位置的权重就变成了 0。  
This makes the attention weight of the final padding position become 0.

## 5. Causal Mask / 因果掩码

在自回归 / autoregressive 场景中，当前位置不应该看到未来位置。  
In autoregressive settings, a position should not see future positions.

这时会用 `causal mask`。  
In that case, we use a causal mask.

In [ ]:
seq_len = 4
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

scores = torch.randn(seq_len, seq_len)
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
weights = F.softmax(masked_scores, dim=-1)

print("causal_mask =\n", causal_mask)
print("weights =\n", weights)

从这个权重矩阵里你会看到：每一行只能看自己和自己之前的位置。  
From this weight matrix, you can see that each row can only attend to itself and the positions before it.

## 6. 一个最小 Self-Attention 模块
## A Minimal Self-Attention Module

下面写一个非常小的单头 self-attention。  
Below we implement a very small single-head self-attention module.

In [ ]:
class SimpleSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        context = weights @ V
        return context, weights


attn = SimpleSelfAttention(d_model=4)
x = torch.randn(2, 5, 4)
context, weights = attn(x)

print("x.shape =", x.shape)
print("context.shape =", context.shape)
print("weights.shape =", weights.shape)

这里的 shape 很关键：  
The shapes here are very important:

- `x.shape == (batch_size, seq_len, d_model)`
- `weights.shape == (batch_size, seq_len, seq_len)`
- `context.shape == (batch_size, seq_len, d_model)`

也就是说，每个位置都会对整条序列分配一组注意力权重。  
That means each position assigns one set of attention weights over the whole sequence.

In [ ]:
# 练习 1 / Exercise 1
# 已知输入 x.shape == (3, 7, 16)
# Given x.shape == (3, 7, 16)
#
# 如果用单头 self-attention，context.shape 和 weights.shape 各是多少？
# For single-head self-attention, what are context.shape and weights.shape?

参考答案 / Reference answer:

- `context.shape == (3, 7, 16)`
- `weights.shape == (3, 7, 7)`

因为每个位置都对 `seq_len=7` 个位置分配权重。  
Because each position assigns attention weights over `seq_len=7` positions.

In [ ]:
# 练习 2 / Exercise 2
# 给定一个 scores 向量 [2.0, 1.0, -1.0]，
# Given a scores vector [2.0, 1.0, -1.0],
# 请用 softmax 算出注意力权重。
# compute the attention weights with softmax.

# scores =
# weights =
# print(weights)

In [ ]:
# 练习 2 参考答案 / Exercise 2 Reference Solution

scores = torch.tensor([2.0, 1.0, -1.0])
weights = F.softmax(scores, dim=-1)
print(weights)

## 7. 和 LSTM 的一个直观对比 / One Intuitive Contrast with LSTM

你可以先记一个足够实用的差别：  
For now, keep one practical difference in mind:

- `LSTM`：按时间步逐步传递信息 / propagates information step by step across time
- `Attention`：一个位置可以直接看所有位置 / one position can directly look at all positions

这不是完整比较，但已经足够建立初始直觉。  
This is not a complete comparison, but it is enough to build an initial intuition.

## 8. 小结 / Summary

这一节最重要的是理解 attention 的三步计算：  
The most important outcome of this notebook is understanding the three-step computation of attention:

1. 点积打分 / dot-product scoring
2. `softmax` 变成权重 / turn scores into weights with `softmax`
3. 对 `V` 做加权求和 / take a weighted sum over `V`

你现在应该能回答 / You should now be able to answer:

1. `Q`、`K`、`V` 各自扮演什么角色？ / What roles do `Q`, `K`, and `V` play?
2. 为什么 attention 里要做 `softmax`？ / Why do we use `softmax` in attention?
3. 为什么要除以 `sqrt(d_k)`？ / Why do we divide by `sqrt(d_k)`?
4. padding mask 和 causal mask 分别解决什么问题？ / What problems do padding masks and causal masks solve?

下一步建议 / Suggested next step:

- 如果继续推进，可以进入 `Transformer Encoder` notebook，把 attention 放进更完整的结构里 / If you continue, the next natural step is a `Transformer Encoder` notebook that places attention into a fuller structure.